# Macro Indicators & Price Elasticity Framework

Analysis of macroeconomic indicators relevant to Starbucks pricing strategy.

**Data sources**:
- FRED: CPI (CPIAUCSL), Average Hourly Earnings (CES0500000003)
- Open-Meteo: Daily temperature for 5 major US cities

**Analysis goals**:
1. CPI trend analysis
2. Real wage index trend
3. Weather patterns and seasonality
4. Price elasticity framework

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
matplotlib.rcParams["figure.figsize"] = (14, 6)
matplotlib.rcParams["figure.dpi"] = 100

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    DATA_DIR = Path("/kaggle/input/starbucks-recommendation-engine")
    if not DATA_DIR.exists():
        DATA_DIR = Path("/kaggle/input/datasets/shiratoriseto/starbucks-recommendation-engine")
else:
    DATA_DIR = Path("../data/raw")

# Load data
macro = pd.read_csv(DATA_DIR / "fred_macro.csv", parse_dates=["date"])
weather = pd.read_csv(DATA_DIR / "weather_daily.csv", parse_dates=["date"])

print(f"FRED macro data: {macro.shape}")
print(f"Weather data: {weather.shape}")
macro.head()

## 1. CPI Trend Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CPI level
axes[0].plot(macro["date"], macro["cpi"], color="steelblue", linewidth=2)
axes[0].set_title("Consumer Price Index (CPI) - All Urban Consumers")
axes[0].set_ylabel("CPI")
axes[0].set_xlabel("Date")
axes[0].grid(True, alpha=0.3)

# CPI YoY change
macro["cpi_yoy"] = macro["cpi"].pct_change(12) * 100
axes[1].bar(macro["date"], macro["cpi_yoy"], color="coral", width=25)
axes[1].axhline(y=2.0, color="gray", linestyle="--", label="2% target")
axes[1].set_title("CPI Year-over-Year Change (%)")
axes[1].set_ylabel("YoY Change (%)")
axes[1].set_xlabel("Date")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"CPI range: {macro['cpi'].min():.1f} - {macro['cpi'].max():.1f}")
print(f"Latest CPI YoY: {macro['cpi_yoy'].dropna().iloc[-1]:.2f}%")

## 2. Real Wage Index Trend

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Nominal wages
axes[0].plot(macro["date"], macro["avg_hourly_earnings"], color="forestgreen", linewidth=2)
axes[0].set_title("Average Hourly Earnings (Nominal)")
axes[0].set_ylabel("USD/hour")
axes[0].set_xlabel("Date")
axes[0].grid(True, alpha=0.3)

# Real wage index (earnings / CPI * 100)
axes[1].plot(macro["date"], macro["real_wage_index"], color="darkorange", linewidth=2)
axes[1].set_title("Real Wage Index (Earnings / CPI × 100)")
axes[1].set_ylabel("Index")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)

# Add trend line
valid = macro.dropna(subset=["real_wage_index"])
x_num = (valid["date"] - valid["date"].min()).dt.days.values
z = np.polyfit(x_num, valid["real_wage_index"].values, 1)
p = np.poly1d(z)
axes[1].plot(valid["date"], p(x_num), "--", color="gray", alpha=0.7, label="Trend")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Nominal wage growth (5yr): {macro['avg_hourly_earnings'].iloc[-1] - macro['avg_hourly_earnings'].iloc[0]:.2f} USD/hr")
print(f"Real wage index change: {macro['real_wage_index'].iloc[-1] - macro['real_wage_index'].dropna().iloc[0]:.4f}")

## 3. Weather Patterns & Seasonality

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Daily temperature by city
for city in weather["city"].unique():
    city_data = weather[weather["city"] == city]
    axes[0].plot(city_data["date"], city_data["temp_mean_f"], alpha=0.7, label=city, linewidth=0.8)

axes[0].set_title("Daily Mean Temperature by City (°F)")
axes[0].set_ylabel("Temperature (°F)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Monthly average across cities
weather["month"] = weather["date"].dt.month
weather["year_month"] = weather["date"].dt.to_period("M")
monthly_avg = weather.groupby(["city", "month"])["temp_mean_f"].mean().reset_index()

for city in monthly_avg["city"].unique():
    city_data = monthly_avg[monthly_avg["city"] == city]
    axes[1].plot(city_data["month"], city_data["temp_mean_f"], marker="o", label=city)

axes[1].set_title("Average Monthly Temperature by City (°F)")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Temperature (°F)")
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Price Elasticity Framework

A framework for estimating price elasticity of demand for Starbucks beverages, combining macro indicators with menu data.

**Approach**: Since we don't have actual sales/demand data, we build a *structural framework* that can be populated with real transaction data later.

The key relationships to model:
- **CPI effect**: As CPI rises, consumers may trade down (switch to cheaper items)
- **Real wage effect**: Higher real wages → more willingness to pay premium
- **Temperature effect**: Hot weather → more cold beverages (Frappuccinos); cold weather → more hot drinks
- **Price-calorie ratio**: Value perception metric that affects demand

In [ ]:
menu = pd.read_csv(DATA_DIR / "starbucks_menu.csv")

# --- Price Elasticity Simulation Framework ---
# Simulate demand response to price changes using standard elasticity formula:
# Q_new / Q_base = (P_new / P_base) ^ elasticity

# Literature-based elasticity estimates for coffee beverages
elasticity_by_category = {
    "Coffee": -0.3,                          # inelastic (daily habit)
    "Classic Espresso Drinks": -0.5,
    "Signature Espresso Drinks": -0.8,
    "Tazo® Tea Drinks": -0.6,
    "Shaken Iced Beverages": -0.9,
    "Smoothies": -1.2,                        # elastic (discretionary)
    "Frappuccino® Blended Coffee": -1.1,
    "Frappuccino® Blended Crème": -1.3,
    "Frappuccino® Light Blended Coffee": -1.0,
}

menu["elasticity"] = menu["category"].map(elasticity_by_category)
assert menu["elasticity"].notna().all(), f"Unmapped categories: {menu[menu['elasticity'].isna()]['category'].unique()}"

# Simulate 5% and 10% price increase impact
for pct in [5, 10]:
    col = f"demand_change_{pct}pct"
    menu[col] = ((1 + pct / 100) ** menu["elasticity"] - 1) * 100

print("Estimated demand change (%) by category for price increases:")
demand_impact = menu.groupby("category")[[
    "elasticity", "demand_change_5pct", "demand_change_10pct"
]].mean().round(2)
demand_impact

In [ ]:
# Visualize elasticity and demand impact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Elasticity by category
demand_impact["elasticity"].sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue"
)
axes[0].set_title("Price Elasticity by Category")
axes[0].set_xlabel("Elasticity (more negative = more elastic)")
axes[0].axvline(x=-1.0, color="red", linestyle="--", alpha=0.5, label="Unit elastic")
axes[0].legend()

# Demand impact comparison
demand_impact[["demand_change_5pct", "demand_change_10pct"]].plot(
    kind="barh", ax=axes[1], color=["coral", "crimson"]
)
axes[1].set_title("Estimated Demand Change for Price Increases")
axes[1].set_xlabel("Demand Change (%)")
axes[1].legend(["5% price increase", "10% price increase"])

plt.tight_layout()
plt.show()

## 5. CPI-Adjusted Affordability Index

Combining CPI trends with menu prices to create an affordability metric over time.

In [ ]:
# Simulate how a Grande Latte's "real" price has changed
# using CPI to deflate a hypothetical price trajectory

base_price = 4.25  # Grande Latte base price in 2021
base_cpi = macro["cpi"].iloc[0]

# Assume Starbucks raises prices roughly in line with food-away-from-home CPI
macro["simulated_latte_price"] = base_price * (macro["cpi"] / base_cpi)
macro["real_latte_price"] = macro["simulated_latte_price"] / (macro["cpi"] / base_cpi)

# Affordability: minutes of work to buy a latte
macro["minutes_to_buy_latte"] = (macro["simulated_latte_price"] / macro["avg_hourly_earnings"]) * 60

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(macro["date"], macro["simulated_latte_price"], color="brown", linewidth=2)
axes[0].set_title("Simulated Grande Latte Price (Nominal, CPI-adjusted)")
axes[0].set_ylabel("Price (USD)")
axes[0].set_xlabel("Date")
axes[0].grid(True, alpha=0.3)

axes[1].plot(macro["date"], macro["minutes_to_buy_latte"], color="purple", linewidth=2)
axes[1].set_title("Minutes of Work to Buy a Grande Latte")
axes[1].set_ylabel("Minutes")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"2021 Q1: {macro['minutes_to_buy_latte'].iloc[0]:.1f} min to buy a latte")
print(f"Latest:  {macro['minutes_to_buy_latte'].iloc[-1]:.1f} min to buy a latte")

## Next Steps

1. **Integrate real transaction/sales data** to calibrate elasticity estimates
2. **Build demand model**: Regression combining price, CPI, real wages, temperature → demand
3. **Seasonal menu optimization**: Use temperature patterns to time seasonal launches
4. **Price sensitivity segmentation**: Cluster customers by price sensitivity using transaction data